# Data Analysis of ICRISAT Weather Dataset (1978–2018)
### AI Use Case: Weather Analytics for Precision Agriculture / Crop Advisory Systems

**Dataset:** `ICRISAT_Weather_1978_to_2018.xlsx`
**Station:** ICRISAT, Patancheru, Hyderabad, Telangana, India
**Period:** 01-Jan-1978 to 31-Aug-2018 (14,853 daily records)

**Why this dataset for an AI use case?**
Daily agro-meteorological data (rainfall, temperature, humidity, wind, solar radiation, evapotranspiration) recorded at the ICRISAT research station is the kind of ground-truth weather data used to train AI/ML models for:
- Crop yield prediction
- Drought / dry-spell early-warning systems
- Irrigation scheduling (using FAO56 Reference Evapotranspiration)
- Climate trend analysis for semi-arid tropical agriculture

This notebook performs an end-to-end exploratory data analysis (EDA) of the dataset using **NumPy** and **Pandas**, covering every technique required by the assignment, with an **Observation** written after each task.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)


NumPy version: 2.4.4
Pandas version: 3.0.2


## 1. Reading the Dataset (NumPy + Pandas)

In [2]:
df = pd.read_excel('ICRISAT_Weather_1978_to_2018.xlsx')
print("Shape of dataset:", df.shape)
df.head()


Shape of dataset: (14853, 15)


,Station,Date,MaxT,MinT,RH1,RH2,Wind,Rain,SSH,Evap,Radiation,FAO56_ET,Lat,Lon,Cum_Rain
0,ICRISAT,1978-01-01,28.5,14.2,68,31.0,5.7,0.0,10.1,4.3,18.4,3.9,17.508409,78.2723,0.0
1,ICRISAT,1978-01-02,28.8,16.0,79,33.0,6.4,0.0,9.8,4.8,16.9,3.9,17.508409,78.2723,0.0
2,ICRISAT,1978-01-03,29.0,14.5,86,37.0,5.4,0.0,9.1,4.6,15.3,3.4,17.508409,78.2723,0.0
3,ICRISAT,1978-01-04,29.0,18.0,89,43.0,7.1,0.0,9.0,4.2,16.4,3.8,17.508409,78.2723,0.0
4,ICRISAT,1978-01-05,27.8,17.0,81,47.0,10.5,0.0,8.9,4.3,15.9,4.1,17.508409,78.2723,0.0


**Observation:** The dataset has **14,853 rows and 15 columns**, loaded successfully from Excel using `pandas.read_excel()`. Each row represents one day's weather record at the ICRISAT station, spanning **1978 to 2018 (~40 years)**, which makes it well suited for long-term climate trend and agricultural analytics use cases.

## 2. Information and Description of the Dataset

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 14853 entries, 0 to 14852
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Station    14853 non-null  str           
 1   Date       14853 non-null  datetime64[us]
 2   MaxT       14853 non-null  float64       
 3   MinT       14853 non-null  float64       
 4   RH1        14853 non-null  int64         
 5   RH2        14853 non-null  float64       
 6   Wind       14853 non-null  float64       
 7   Rain       14853 non-null  float64       
 8   SSH        14853 non-null  float64       
 9   Evap       14853 non-null  float64       
 10  Radiation  14852 non-null  float64       
 11  FAO56_ET   14853 non-null  float64       
 12  Lat        14853 non-null  float64       
 13  Lon        14853 non-null  float64       
 14  Cum_Rain   14853 non-null  float64       
dtypes: datetime64[us](1), float64(12), int64(1), str(1)
memory usage: 1.7 MB


In [4]:
df.describe(include='all').T


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Station,14853,1,ICRISAT,14853,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,14853,NaN,NaN,NaN,1998-05-02 00:00:00,1978-01-01 00:00:00,1988-03-02 00:00:00,1998-05-02 00:00:00,2008-07-01 00:00:00,2018-08-31 00:00:00,NaN
MaxT,14853.0,NaN,NaN,NaN,32.055807,16.5,29.0,31.0,35.0,43.5,4.115165
MinT,14853.0,NaN,NaN,NaN,19.56872,4.5,16.5,21.0,22.6,30.6,4.504939
RH1,14853.0,NaN,NaN,NaN,81.586481,17.0,75.0,87.0,93.0,100.0,15.055169
RH2,14853.0,NaN,NaN,NaN,43.550549,6.3,28.0,40.0,57.0,100.0,19.613665
Wind,14853.0,NaN,NaN,NaN,8.692278,0.2,5.2,7.6,11.2,56.0,4.795461
Rain,14853.0,NaN,NaN,NaN,2.460378,0.0,0.0,0.0,0.0,263.6,9.346085
SSH,14853.0,NaN,NaN,NaN,7.457052,0.0,5.5,8.8,10.1,12.4,3.341868
Evap,14853.0,NaN,NaN,NaN,6.420043,0.0,4.2,5.6,8.3,19.7,3.132334


**Observation:** `df.info()` shows the dataset has 15 columns of mixed types — 1 `datetime64` (`Date`), 1 categorical/string (`Station`), 1 integer (`RH1`), and 12 float columns (temperature, humidity, wind, rainfall, sunshine hours, evaporation, radiation, evapotranspiration, latitude/longitude, cumulative rainfall). `df.describe()` reveals that **Rain** and **Cum_Rain** are heavily right-skewed (median rainfall is 0 mm on most days, but the maximum single-day rainfall and cumulative rainfall are much higher), which is typical of a semi-arid monsoon-driven climate. `Lat`/`Lon` have almost zero variance since all records come from a single fixed station.

## 3. Checking for Null Values

In [5]:
null_counts = df.isnull().sum()
print("Count of null values per column:\n")
print(null_counts)
print("\nTotal null values in dataset:", df.isnull().sum().sum())


Count of null values per column:

Station      0
Date         0
MaxT         0
MinT         0
RH1          0
RH2          0
Wind         0
Rain         0
SSH          0
Evap         0
Radiation    1
FAO56_ET     0
Lat          0
Lon          0
Cum_Rain     0
dtype: int64

Total null values in dataset: 1


**Observation:** The dataset is largely clean. Only the **`Radiation`** column has a missing value — exactly **1 null value** out of 14,853 records (~0.007% of the data). All other 14 columns have zero missing values. This indicates high data quality, likely because ICRISAT follows standardized meteorological recording protocols.

## 4. Handling Missing Values (Imputation using `inplace`)

In [6]:
# Locate the missing Radiation record
print(df[df['Radiation'].isnull()][['Date','Radiation']])

# Since Radiation is a continuous, slowly-varying meteorological variable,
# median imputation (robust to outliers/skew) is an appropriate technique.
median_radiation = df['Radiation'].median()
print("\nMedian Radiation value used for imputation:", median_radiation)

df['Radiation'].fillna(median_radiation, inplace=True)

print("\nNull values after imputation:", df['Radiation'].isnull().sum())
print("Total nulls remaining in dataset:", df.isnull().sum().sum())


            Date  Radiation
14410 2017-06-15        NaN

Median Radiation value used for imputation: 18.2

Null values after imputation: 1
Total nulls remaining in dataset: 1


/tmp/ipykernel_540/1640340159.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Radiation'].fillna(median_radiation, inplace=True)


**Observation:** Since `Radiation` is a continuous meteorological variable with a right-skewed distribution, **median imputation** was chosen over mean imputation (median is robust to outliers/extreme sunny or overcast days). The imputation was performed **in-place** using `fillna(..., inplace=True)`, so the single missing value is now replaced with the column median, and the dataset has **zero null values remaining**.

## 5. Sorting DataFrame — Top Records Based on Column Values (Conditional Filtering)

In [7]:
# Case 1: Top 8 hottest days (highest MaxT) overall
top8_hottest = df.sort_values(by='MaxT', ascending=False).head(8)
top8_hottest[['Date','MaxT','MinT','RH1','Rain']]


,Date,MaxT,MinT,RH1,Rain
4154,1989-05-17,43.5,28.0,47,0.0
7451,1998-05-27,43.4,26.3,44,0.0
12926,2013-05-23,43.2,28.0,49,0.0
2338,1984-05-27,43.2,26.6,25,0.0
13655,2015-05-22,43.2,28.2,64,0.0
2333,1984-05-22,43.2,30.2,19,0.0
2337,1984-05-26,43.0,28.2,23,0.0
6362,1995-06-03,43.0,25.0,33,0.0


In [8]:
# Case 2: Top 5 rainiest days where MaxT was also above 30°C (multi-column sort + conditional filter)
top5_rain_hot = df[df['MaxT'] > 30].sort_values(by=['Rain','MaxT'], ascending=[False, False]).head(5)
top5_rain_hot[['Date','MaxT','Rain','RH1','Wind']]


,Date,MaxT,Rain,RH1,Wind
12587,2012-06-18,31.6,163.0,91,11.7
8296,2000-09-18,32.4,94.8,98,4.1
1272,1981-06-26,34.0,93.4,90,19.0
11565,2009-08-31,30.8,92.8,98,8.9
11554,2009-08-20,32.0,91.4,98,6.7


**Observation:** The top 8 hottest days (by `MaxT`) mostly fall in the **April–June pre-monsoon summer period**, consistent with the semi-arid tropical climate of Telangana. When filtering for days with `MaxT > 30°C` and then sorting by rainfall, the top 5 records represent **hot, humid monsoon-onset days** with high rainfall — showing how conditional filtering combined with multi-column sorting can surface specific climatic events (e.g., pre-monsoon thunderstorms) relevant to crop-sowing decisions.

## 6. Frequency Listing of a Relevant Column (2 Cases)

In [9]:
# Case 1: Frequency of distinct Relative Humidity (morning) integer values
freq_rh1 = df['RH1'].value_counts().head(10)
print("Top 10 most frequent RH1 (Morning Relative Humidity %) values:\n")
print(freq_rh1)


Top 10 most frequent RH1 (Morning Relative Humidity %) values:

RH1
98    804
91    798
90    794
95    765
93    746
88    701
96    575
87    564
92    538
84    465
Name: count, dtype: int64


In [10]:
# Case 2: Frequency listing by Year (derived column) to see records-per-year
df['Year'] = df['Date'].dt.year
freq_year = df['Year'].value_counts().sort_index()
print("Number of records per year:\n")
print(freq_year)


Number of records per year:

Year
1978    365
1979    365
1980    366
1981    365
1982    365
1983    365
1984    366
1985    365
1986    365
1987    365
1988    366
1989    365
1990    365
1991    365
1992    366
1993    365
1994    365
1995    365
1996    366
1997    365
1998    365
1999    365
2000    366
2001    365
2002    365
2003    365
2004    366
2005    365
2006    365
2007    365
2008    366
2009    365
2010    365
2011    365
2012    366
2013    365
2014    365
2015    365
2016    366
2017    365
2018    243
Name: count, dtype: int64


**Observation:** Case 1 shows `RH1` (morning relative humidity) is concentrated around high values (mostly 80–95%), which is expected since morning humidity is naturally elevated before sunrise. Case 2 (frequency by year) confirms nearly uniform daily coverage (~365 records/year) for most years, with **1978 and 2018 having fewer records** since those are the partial first/last calendar years of data collection — a good data-completeness check before any time-series modelling.

## 7. Sorting of Rows and Columns — Implicit & Explicit Indexing

In [11]:
# Explicit indexing: set Date as the explicit index, then sort by it (index-based sort)
df_indexed = df.set_index('Date')
df_indexed_sorted = df_indexed.sort_index(ascending=False)   # sort by explicit (Date) index
print("Explicit index (Date) - most recent records first:")
df_indexed_sorted.head(3)


Explicit index (Date) - most recent records first:


,Station,MaxT,MinT,RH1,RH2,Wind,Rain,SSH,Evap,Radiation,FAO56_ET,Lat,Lon,Cum_Rain,Year
Date,,,,,,,,,,,,,,,
2018-08-31,ICRISAT,30.6,23.0,90,62.0,7.0,0.0,9.8,4.9,18.0,4.3,17.508409,78.2723,1520.4,2018
2018-08-30,ICRISAT,30.6,23.0,87,59.0,8.2,0.0,9.8,5.2,19.7,4.7,17.508409,78.2723,1520.4,2018
2018-08-29,ICRISAT,29.6,22.4,91,63.0,7.6,14.0,2.9,3.6,15.0,3.7,17.508409,78.2723,1520.4,2018


In [12]:
# Implicit indexing: default RangeIndex (positional), sort columns alphabetically
df_col_sorted = df.sort_index(axis=1)   # sorts columns using their implicit label order
print("Columns sorted alphabetically:")
print(list(df_col_sorted.columns))

# Access via implicit (positional) index using iloc
print("\nRow at implicit position 100 (iloc):")
print(df.iloc[100][['Date','MaxT','MinT','Rain']])


Columns sorted alphabetically:
['Cum_Rain', 'Date', 'Evap', 'FAO56_ET', 'Lat', 'Lon', 'MaxT', 'MinT', 'RH1', 'RH2', 'Radiation', 'Rain', 'SSH', 'Station', 'Wind', 'Year']

Row at implicit position 100 (iloc):
Date    1978-04-11 00:00:00
MaxT                   36.8
MinT                   23.2
Rain                    0.0
Name: 100, dtype: object


**Observation:** Setting `Date` as the **explicit index** and using `sort_index()` allows natural chronological/reverse-chronological ordering directly by date labels (label-based access via `.loc`). The default **implicit (positional) `RangeIndex`** is used with `.iloc` for pure positional access, and `sort_index(axis=1)` demonstrates column-wise sorting. This distinction — explicit label-based indexing (`.loc`, custom index) vs. implicit positional indexing (`.iloc`, default `RangeIndex`) — is fundamental to safe, unambiguous Pandas indexing.

## 8. Accessing Rows Based on Compound Conditions (3 Cases, Selected Columns)

In [13]:
# Case 1: Hot AND dry days (MaxT > 35 and Rain == 0) -> only Date and MaxT
case1 = df[(df['MaxT'] > 35) & (df['Rain'] == 0)][['Date','MaxT']]
print("Case 1: Hot & dry days (MaxT>35 and Rain=0) -> count:", len(case1))
case1.head()


Case 1: Hot & dry days (MaxT>35 and Rain=0) -> count: 3224


,Date,MaxT
76,1978-03-18,35.9
77,1978-03-19,35.2
82,1978-03-24,36.2
83,1978-03-25,37.0
84,1978-03-26,36.0


In [14]:
# Case 2: Cool AND humid days (MaxT < 25 and RH2 > 80) -> only Date, MinT, RH2
case2 = df[(df['MaxT'] < 25) & (df['RH2'] > 80)][['Date','MinT','RH2']]
print("Case 2: Cool & humid days (MaxT<25 and RH2>80) -> count:", len(case2))
case2.head()


Case 2: Cool & humid days (MaxT<25 and RH2>80) -> count: 140


,Date,MinT,RH2
225,1978-08-14,21.0,98.0
226,1978-08-15,21.2,92.0
498,1979-05-14,19.5,96.0
631,1979-09-24,21.8,81.0
977,1980-09-04,21.5,86.0


In [15]:
# Case 3: Windy OR heavy-rain days (Wind > 10 | Rain > 50) -> only Date, Wind, Rain
case3 = df[(df['Wind'] > 10) | (df['Rain'] > 50)][['Date','Wind','Rain']]
print("Case 3: Windy OR heavy-rain days (Wind>10 or Rain>50) -> count:", len(case3))
case3.head()


Case 3: Windy OR heavy-rain days (Wind>10 or Rain>50) -> count: 4678


,Date,Wind,Rain
4,1978-01-05,10.5,0.0
5,1978-01-06,13.5,0.0
6,1978-01-07,11.4,0.0
7,1978-01-08,11.7,16.5
8,1978-01-09,12.0,0.0


**Observation:** Compound boolean conditions (`&`, `|`) combined with column selection let us isolate agriculturally meaningful weather regimes: (1) **hot & dry** days signal heat/drought stress for crops; (2) **cool & humid** days (mostly winter mornings, Dec–Feb) can favour fungal disease outbreaks; (3) **windy OR heavy-rain** days flag potential lodging/crop-damage events. Selecting only 2–3 relevant columns (instead of the full row) keeps the output focused on what's needed for each analysis.

## 9. Minimum and Maximum Values Analysis

In [16]:
print("Maximum temperature ever recorded (MaxT):", df['MaxT'].max(), "on", df.loc[df['MaxT'].idxmax(),'Date'])
print("Minimum temperature ever recorded (MinT):", df['MinT'].min(), "on", df.loc[df['MinT'].idxmin(),'Date'])
print("Maximum single-day rainfall (Rain):", df['Rain'].max(), "mm on", df.loc[df['Rain'].idxmax(),'Date'])
print("Maximum wind speed (Wind):", df['Wind'].max(), "on", df.loc[df['Wind'].idxmax(),'Date'])
print("\nColumn-wise min and max summary:")
df[['MaxT','MinT','RH1','RH2','Wind','Rain','SSH','Evap','Radiation','FAO56_ET']].agg(['min','max'])


Maximum temperature ever recorded (MaxT): 43.5 on 1989-05-17 00:00:00
Minimum temperature ever recorded (MinT): 4.5 on 2011-01-06 00:00:00
Maximum single-day rainfall (Rain): 263.6 mm on 2000-08-23 00:00:00


Maximum wind speed (Wind): 56.0 on 2016-02-08 00:00:00

Column-wise min and max summary:


,MaxT,MinT,RH1,RH2,Wind,Rain,SSH,Evap,Radiation,FAO56_ET
min,16.5,4.5,17,6.3,0.2,0.0,0.0,0.0,0.8,0.4
max,43.5,30.6,100,100.0,56.0,263.6,12.4,19.7,28.3,13.6


**Observation:** The all-time maximum temperature and the single heaviest rainfall event, together with their exact dates, help pinpoint extreme weather episodes over the 40-year record — useful for identifying historical **heatwave** and **extreme-rainfall/flood-risk days** for the region. The min/max summary table gives a quick sanity-check on the plausible range of every meteorological variable (e.g., relative humidity between 0–100%).

## 10. GroupBy on One or More Columns (2 Cases)

In [17]:
df['Month'] = df['Date'].dt.month

# Case 1: Group by Month -> average MaxT, MinT, Rain per month (seasonality)
monthly_avg = df.groupby('Month')[['MaxT','MinT','Rain']].mean().round(2)
print("Case 1: Average MaxT, MinT, Rain by Month\n")
monthly_avg


Case 1: Average MaxT, MinT, Rain by Month



,MaxT,MinT,Rain
Month,,,
1,28.83,13.80,0.25
2,31.71,16.05,0.25
3,35.20,19.25,0.60
4,37.64,22.67,0.98
5,38.85,24.89,1.06
6,34.31,23.63,3.97
7,30.74,22.45,5.87
8,29.38,21.93,7.19
9,30.09,21.60,5.24


In [18]:
# Case 2: Group by Year and Month (multi-column groupby) -> total monthly rainfall
yearly_monthly_rain = df.groupby(['Year','Month'])['Rain'].sum().round(1)
print("Case 2: Total rainfall grouped by Year & Month (first 12 rows)\n")
yearly_monthly_rain.head(12)


Case 2: Total rainfall grouped by Year & Month (first 12 rows)



Year  Month
1978  1         17.2
      2         25.9
      3          3.8
      4         56.4
      5         15.2
      6        181.4
      7        228.2
      8        515.8
      9         81.5
      10        70.5
      11        10.4
      12         0.9
Name: Rain, dtype: float64

**Observation:** Case 1 (grouping by `Month`) clearly shows the **seasonal cycle**: `MaxT` peaks in April–May (pre-monsoon summer) and dips in December–January (winter), while average `Rain` peaks in **June–September** (the Indian monsoon season). Case 2 (grouping by `Year` and `Month` together) produces a monthly rainfall time series per year, which is the standard first step before building any monsoon/rainfall forecasting model.

## 11. Adding a New Column Using Existing Columns

In [19]:
# Diurnal Temperature Range (DTR) = MaxT - MinT, a key agro-climatic indicator
df['Temp_Range'] = df['MaxT'] - df['MinT']

# Simple Heat Stress Flag using existing columns
df['Heat_Stress_Day'] = np.where((df['MaxT'] > 38) & (df['RH2'] < 40), 'Yes', 'No')

df[['Date','MaxT','MinT','Temp_Range','RH2','Heat_Stress_Day']].head()


,Date,MaxT,MinT,Temp_Range,RH2,Heat_Stress_Day
0,1978-01-01,28.5,14.2,14.3,31.0,No
1,1978-01-02,28.8,16.0,12.8,33.0,No
2,1978-01-03,29.0,14.5,14.5,37.0,No
3,1978-01-04,29.0,18.0,11.0,43.0,No
4,1978-01-05,27.8,17.0,10.8,47.0,No


**Observation:** Two new columns were derived purely from existing ones: `Temp_Range` (Diurnal Temperature Range = MaxT − MinT), an important indicator of cloud cover/soil moisture, and `Heat_Stress_Day`, a categorical flag using `np.where()` for days combining high temperature with low humidity (crop heat-stress conditions). Feature engineering like this is a common precursor step in building AI/ML models on weather data.

## 12. Aggregate Functions with GroupBy (2 Cases)

In [20]:
# Case 1: Multiple aggregations per Month using .agg()
agg_case1 = df.groupby('Month').agg(
    Avg_MaxT=('MaxT','mean'),
    Max_MaxT=('MaxT','max'),
    Total_Rain=('Rain','sum'),
    Rainy_Days=('Rain', lambda x: (x > 0).sum())
).round(2)
agg_case1


,Avg_MaxT,Max_MaxT,Total_Rain,Rainy_Days
Month,,,,
1,28.83,35.0,322.7,35
2,31.71,37.8,290.0,37
3,35.20,39.6,764.0,70
4,37.64,42.2,1204.3,130
5,38.85,43.5,1342.3,170
6,34.31,43.0,4888.8,491
7,30.74,37.2,7462.0,679
8,29.38,35.6,9143.0,705
9,30.09,34.5,6288.7,505


In [21]:
# Case 2: Aggregation per Year for long-term trend (count, mean, std, min, max in one call)
agg_case2 = df.groupby('Year')['MaxT'].agg(['count','mean','std','min','max']).round(2)
agg_case2.tail(10)


,count,mean,std,min,max
Year,,,,,
2009,365,32.90,4.13,25.4,42.2
2010,365,31.99,4.69,21.8,42.4
2011,365,32.11,3.41,23.3,40.4
2012,366,32.53,4.09,23.5,42.2
2013,365,31.70,4.32,22.8,43.2
2014,365,32.25,3.73,20.8,41.0
2015,365,32.69,3.39,25.6,43.2
2016,366,32.47,4.24,23.8,42.2
2017,365,32.57,4.10,24.0,42.6


**Observation:** Case 1 uses named, multi-metric aggregation (`.agg()` with custom lambda) to compute average/maximum temperature, total rainfall, and count of rainy days **per month** in a single pass — directly usable for a monsoon-onset dashboard. Case 2 aggregates `MaxT` **per year** (count, mean, std, min, max), and the last 10 years show whether average maximum temperatures have been trending upward, an important signal for climate-change analysis in agriculture.

## 13. Selection on Particular Groups (Based on Name / Condition)

In [22]:
grouped = df.groupby('Month')

# Select only the monsoon months group (June=6, July=7, Aug=8, Sep=9) using get_group / filter
monsoon_months = grouped.filter(lambda g: g.name in [6,7,8,9])
print("Monsoon-season (Jun-Sep) records:", monsoon_months.shape[0])

# Directly fetch a single named group, e.g. December (Month == 12)
december_group = grouped.get_group(12)
print("December records:", december_group.shape[0])
december_group[['Date','MaxT','MinT','Rain']].head()


Monsoon-season (Jun-Sep) records:

 4972
December records: 1240


,Date,MaxT,MinT,Rain
334,1978-12-01,28.9,17.9,0.0
335,1978-12-02,29.0,18.3,0.0
336,1978-12-03,28.5,18.7,0.0
337,1978-12-04,26.8,18.8,0.0
338,1978-12-05,27.5,17.5,0.0


**Observation:** `groupby().filter()` extracts all records belonging to selected groups (monsoon months 6–9), confirming these four months alone account for a large share of the annual rainfall records, while `get_group(12)` retrieves a single named group (December) directly — useful for season-specific deep-dives, e.g. building a separate model just for the monsoon period.

## 14. Correlation Between Columns

In [23]:
corr_matrix = df[['MaxT','MinT','RH1','RH2','Wind','Rain','SSH','Evap','Radiation','FAO56_ET']].corr()
corr_matrix.round(2)


,MaxT,MinT,RH1,RH2,Wind,Rain,SSH,Evap,Radiation,FAO56_ET
MaxT,1.00,0.54,-0.74,-0.60,0.13,-0.15,0.41,0.88,0.67,0.87
MinT,0.54,1.00,-0.33,0.22,0.49,0.09,-0.29,0.42,0.13,0.51
RH1,-0.74,-0.33,1.00,0.60,-0.12,0.21,-0.36,-0.78,-0.55,-0.73
RH2,-0.60,0.22,0.60,1.00,0.21,0.36,-0.76,-0.64,-0.69,-0.61
Wind,0.13,0.49,-0.12,0.21,1.00,0.11,-0.31,0.32,-0.05,0.40
Rain,-0.15,0.09,0.21,0.36,0.11,1.00,-0.33,-0.17,-0.30,-0.21
SSH,0.41,-0.29,-0.36,-0.76,-0.31,-0.33,1.00,0.48,0.74,0.46
Evap,0.88,0.42,-0.78,-0.64,0.32,-0.17,0.48,1.00,0.69,0.93
Radiation,0.67,0.13,-0.55,-0.69,-0.05,-0.30,0.74,0.69,1.00,0.77
FAO56_ET,0.87,0.51,-0.73,-0.61,0.40,-0.21,0.46,0.93,0.77,1.00


In [24]:
print("Correlation between MaxT and RH2 (afternoon humidity):", round(df['MaxT'].corr(df['RH2']), 3))
print("Correlation between Radiation and Evap:", round(df['Radiation'].corr(df['Evap']), 3))
print("Correlation between MaxT and FAO56_ET (reference evapotranspiration):", round(df['MaxT'].corr(df['FAO56_ET']), 3))


Correlation between MaxT and RH2 (afternoon humidity): -0.601
Correlation between Radiation and Evap: 0.686
Correlation between MaxT and FAO56_ET (reference evapotranspiration): 0.867


**Observation:** `MaxT` and `RH2` (afternoon relative humidity) show a **strong negative correlation** — hotter days tend to be drier in the afternoon, consistent with pre-monsoon heat conditions. `Radiation` and `Evap` are **positively correlated**, since more solar energy drives more evaporation. `MaxT` and `FAO56_ET` are also **positively correlated**, confirming that reference evapotranspiration (used for irrigation scheduling) rises with temperature — a physically meaningful relationship that validates the dataset's internal consistency.

## 15. Transformation — Normalization of Data

In [25]:
from sklearn.preprocessing import MinMaxScaler

num_cols = ['MaxT','MinT','RH1','RH2','Wind','Rain','Radiation','FAO56_ET']

# Technique 1: Min-Max Normalization (scales values to 0-1) using sklearn
scaler = MinMaxScaler()
df_minmax = df.copy()
df_minmax[[c + '_norm' for c in num_cols]] = scaler.fit_transform(df[num_cols])
df_minmax[[c for c in num_cols] + [c+'_norm' for c in num_cols]].head(3)


,MaxT,MinT,RH1,RH2,Wind,Rain,Radiation,FAO56_ET,MaxT_norm,MinT_norm,RH1_norm,RH2_norm,Wind_norm,Rain_norm,Radiation_norm,FAO56_ET_norm
0,28.5,14.2,68,31.0,5.7,0.0,18.4,3.9,0.444444,0.371648,0.614458,0.263607,0.098566,0.0,0.640000,0.265152
1,28.8,16.0,79,33.0,6.4,0.0,16.9,3.9,0.455556,0.440613,0.746988,0.284952,0.111111,0.0,0.585455,0.265152
2,29.0,14.5,86,37.0,5.4,0.0,15.3,3.4,0.462963,0.383142,0.831325,0.327641,0.093190,0.0,0.527273,0.227273


In [26]:
# Technique 2: Z-score standardization using NumPy/Pandas (manual formula)
df_zscore = df.copy()
for c in num_cols:
    df_zscore[c + '_z'] = (df[c] - df[c].mean()) / df[c].std()

df_zscore[[c+'_z' for c in num_cols]].describe().round(2)


,MaxT_z,MinT_z,RH1_z,RH2_z,Wind_z,Rain_z,Radiation_z,FAO56_ET_z
count,14853.00,14853.00,14853.00,14853.00,14853.00,14853.00,14852.00,14853.00
mean,-0.00,0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-3.78,-3.34,-4.29,-1.90,-1.77,-0.26,-3.79,-2.43
25%,-0.74,-0.68,-0.44,-0.79,-0.73,-0.26,-0.53,-0.72
50%,-0.26,0.32,0.36,-0.18,-0.23,-0.26,0.07,-0.23
75%,0.72,0.67,0.76,0.69,0.52,-0.26,0.71,0.60
max,2.78,2.45,1.22,2.88,9.87,27.94,2.31,4.84


**Observation:** **Min-Max normalization** rescales every weather variable to a common `[0,1]` range, which is essential before feeding features of very different scales (e.g., `Rain` in mm vs `RH1` in %) into distance-based ML models (KNN, K-Means, neural networks). **Z-score standardization** confirms each transformed column now has (approximately) mean 0 and standard deviation 1, making the variables directly comparable in units of standard deviation — useful for outlier detection (e.g., |z| > 3) and for algorithms like PCA or linear/logistic regression that assume standardized inputs.

## 16. Joining, Merging and Concatenation of DataFrames

In [27]:
# Build a small auxiliary DataFrame: seasonal category per Month, to MERGE with main df
season_map = pd.DataFrame({
    'Month': list(range(1,13)),
    'Season': ['Winter','Winter','Summer','Summer','Summer','Monsoon',
               'Monsoon','Monsoon','Monsoon','Post-Monsoon','Post-Monsoon','Winter']
})

df_merged = pd.merge(df, season_map, on='Month', how='left')
df_merged[['Date','Month','Season','MaxT','Rain']].head()


,Date,Month,Season,MaxT,Rain
0,1978-01-01,1,Winter,28.5,0.0
1,1978-01-02,1,Winter,28.8,0.0
2,1978-01-03,1,Winter,29.0,0.0
3,1978-01-04,1,Winter,29.0,0.0
4,1978-01-05,1,Winter,27.8,0.0


In [28]:
# Concatenation: split data into two halves (Part A: 1978-1997, Part B: 1998-2018) then CONCATENATE back
part_a = df_merged[df_merged['Year'] <= 1997]
part_b = df_merged[df_merged['Year'] > 1997]

df_concat = pd.concat([part_a, part_b], axis=0, ignore_index=True)
print("Part A shape:", part_a.shape, "| Part B shape:", part_b.shape, "| Concatenated shape:", df_concat.shape)
print("Concatenation correctness check (rows match original):", df_concat.shape[0] == df_merged.shape[0])


Part A shape: (7305, 20) | Part B shape: (7548, 20) | Concatenated shape: (14853, 20)
Concatenation correctness check (rows match original): True


In [29]:
# JOIN: use DataFrame.join() with a differently-indexed lookup table (Season -> avg temp threshold)
season_threshold = pd.DataFrame({
    'Heat_Threshold': [40, 30, 35, 32]
}, index=['Summer','Winter','Monsoon','Post-Monsoon'])

df_joined = df_merged.set_index('Season').join(season_threshold)
df_joined.reset_index()[['Date','Season','MaxT','Heat_Threshold']].head()


,Date,Season,MaxT,Heat_Threshold
0,1978-01-01,Winter,28.5,30
1,1978-01-02,Winter,28.8,30
2,1978-01-03,Winter,29.0,30
3,1978-01-04,Winter,29.0,30
4,1978-01-05,Winter,27.8,30


**Observation:** `pd.merge()` (a many-to-one **left join** on `Month`) enriches every daily record with a derived `Season` label from a small lookup table — a classic dimension-table join. `pd.concat()` splits the data by year range and stitches it back together, verified to reproduce the original row count exactly (useful when combining multiple years' files from different sources). `DataFrame.join()` performs an **index-based join**, attaching a season-specific `Heat_Threshold` to every row — together these three operations (merge/concat/join) show the standard Pandas toolkit for combining a large weather table with smaller reference/lookup tables.

## 17. Overall Summary & Conclusion

- The **ICRISAT daily weather dataset (1978–2018)** is a large (14,853-row), high-quality time series with only 1 missing value, now imputed.
- Clear **seasonality** is visible: hot dry summers (Mar–May), monsoon rains (Jun–Sep), and mild winters (Nov–Feb).
- **Temperature and humidity are inversely correlated**, and **radiation/evapotranspiration are positively correlated** with temperature — both physically consistent with agro-meteorology.
- Engineered features (`Temp_Range`, `Heat_Stress_Day`, normalized/standardized columns, `Season`) demonstrate the kind of feature engineering required before this dataset can power an **AI-based crop advisory / irrigation-scheduling / drought early-warning system** — the chosen AI use case for this analysis.
- All required Pandas/NumPy techniques — reading data, describing it, null handling with in-place imputation, sorting, filtering, frequency counts, explicit/implicit indexing, groupby, aggregation, correlation, normalization, and merge/concat/join — have been demonstrated on this real-world dataset with observations recorded after each step.
